# 00 — Google Colab bootstrap and smoke test

Notebook นี้ clone/checkout source, ติดตั้ง candidate lock, import package และบันทึก environment telemetry โดยไม่ mount Google Drive และไม่โหลด model weights

> Privacy: ใช้เฉพาะ synthetic/public data. อย่าอัปโหลด invoice จริงหรือข้อมูล PII เข้า managed Colab runtime.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get('INVOICE_AUDITOR_REPO', 'https://github.com/Sayomphon/Multimodal_Invoice_Auditor.git')
APP_REF = os.environ.get('INVOICE_AUDITOR_REF', 'main')
PROJECT_ROOT = Path('/content/Multimodal_Invoice_Auditor')
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
git_env = os.environ.copy()
if GITHUB_TOKEN:
    git_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'Authorization: Bearer {GITHUB_TOKEN}'})

if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(PROJECT_ROOT)], check=True, env=git_env)
if not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'Existing path is not a Git checkout: {PROJECT_ROOT}')
if subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Colab checkout contains local changes; use a clean runtime or clean checkout')
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth=1', 'origin', APP_REF], check=True, env=git_env)
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
APP_COMMIT = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if len(APP_COMMIT) != 40:
    raise RuntimeError('Application commit did not resolve to a full SHA')
os.chdir(PROJECT_ROOT)
print({'project_root': str(PROJECT_ROOT), 'application_commit': APP_COMMIT})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-colab.lock'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', '-e', '.'], check=True)

from invoice_auditor import __version__
from invoice_auditor.runtime import detect_runtime

runtime = detect_runtime(disk_path=PROJECT_ROOT)
print({'invoice_auditor': __version__, **runtime.model_dump(mode='json')})

In [ ]:
from datetime import UTC, datetime
from invoice_auditor.io_utils import atomic_write_json

BOOTSTRAP_RUN_ID = f"bootstrap-{datetime.now(UTC):%Y%m%dT%H%M%SZ}"
BOOTSTRAP_DIR = PROJECT_ROOT / 'reports' / 'colab' / BOOTSTRAP_RUN_ID
environment_path = atomic_write_json(BOOTSTRAP_DIR / 'environment.json', runtime.model_dump(mode='json'))
print(environment_path)
if not runtime.cuda_available:
    print('GPU not detected: select Runtime > Change runtime type > GPU before notebook 02.')